# ☕ Ćwiczenie 5: Spark SQL i DataFrames — Sieć kawiarni

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 5 (Studia stacjonarne, Semestr Zimowy 2026/2027)
**Szacowany czas:** ⏱️ 90 minut
**Poziom trudności:** ⭐⭐ (Średni)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Rejestrować tymczasowe widoki i pisać zapytania `spark.sql()`
2. Wykonywać agregacje (`groupBy`, `agg`) i porównywać je z odpowiednikiem SQL
3. Łączyć dane transakcyjne z tabelą wymiarową (`join`, `broadcast`)
4. Analizować dane czasowe (godzina, miesiąc) i liczyć wzrosty miesiąc do miesiąca
5. Wskazywać najlepiej sprzedające się produkty i metody płatności per sklep/miasto

## 📋 Wymagania wstępne
- Podstawy PySpark (RDD, transformacje, akcje) — Wykład 4
- Podstawy Spark SQL i DataFrame API — Wykład 5

---
> 💡 **Tip:** Uruchamiaj każdą komórkę przez `Shift+Enter` lub przycisk ▶️.
> Jeśli utkniesz, sprawdź komórki z podpowiedziami (💡 Hint) albo zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — Run this cell first!

!pip install -q pyspark

import random
import datetime as dt

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("CW05-SiecKawiarni")
    .master("local[*]")
    .getOrCreate()
)

random.seed(42)
np.random.seed(42)

print(f"✅ Setup complete! Spark {spark.version} gotowy do parzenia danych ☕🚀")

## 📖 Szybki refresher

Pracujemy dziś z danymi transakcyjnymi **sieci kawiarni** — 15 lokali w kilku miastach Polski, ~100 000 transakcji.

| Koncept | Opis | Przykład |
|---------|------|---------|
| `createOrReplaceTempView` | Rejestruje DataFrame jako tabelę SQL | `df.createOrReplaceTempView("transakcje")` |
| `spark.sql()` | Wykonuje zapytanie SQL na widokach | `spark.sql("SELECT * FROM transakcje")` |
| `groupBy().agg()` | Agregacje w DataFrame API | `df.groupBy("miasto").agg(F.sum("cena"))` |
| `broadcast()` | Wymusza kopiowanie małej tabeli do każdego executora | `df.join(F.broadcast(mala_tabela), "klucz")` |
| `Window` | Operacje analityczne bez redukcji liczby wierszy | `F.rank().over(Window.partitionBy(...))` |

In [ ]:
# @title 📊 Generujemy dane — transakcje sieci kawiarni

MIASTA = ["Warszawa", "Krakow", "Wroclaw", "Poznan", "Gdansk"]
REGIONY = {
    "Warszawa": "Mazowieckie",
    "Krakow": "Malopolskie",
    "Wroclaw": "Dolnoslaskie",
    "Poznan": "Wielkopolskie",
    "Gdansk": "Pomorskie",
}

# 3 sklepy na miasto = 15 sklepów łącznie
sklepy_lista = []
sklep_do_miasta = {}
for i, miasto in enumerate(MIASTA):
    for j in range(3):
        sklep_id = f"SKL{i * 3 + j + 1:02d}"
        sklep_do_miasta[sklep_id] = miasto
        sklepy_lista.append({
            "sklep": sklep_id,
            "miasto": miasto,
            "region": REGIONY[miasto],
            "powierzchnia_m2": random.randint(35, 120),
            "data_otwarcia": dt.date(2020, 1, 1) + dt.timedelta(days=random.randint(0, 1800)),
        })

sklepy_pdf = pd.DataFrame(sklepy_lista)
sklepy = spark.createDataFrame(sklepy_pdf)
sklepy.show()

In [ ]:
# @title 📊 Generujemy ~100 000 transakcji

PRODUKTY = [
    # (produkt, kategoria, cena_bazowa)
    ("Espresso", "kawa", 8.0),
    ("Cappuccino", "kawa", 12.0),
    ("Latte", "kawa", 13.0),
    ("Americano", "kawa", 9.0),
    ("Flat White", "kawa", 14.0),
    ("Herbata czarna", "herbata", 7.0),
    ("Herbata zielona", "herbata", 7.5),
    ("Herbata owocowa", "herbata", 8.0),
    ("Croissant", "przekaska", 9.0),
    ("Sernik", "przekaska", 14.0),
    ("Muffin", "przekaska", 10.0),
    ("Kanapka", "przekaska", 15.0),
    ("Woda", "inne", 6.0),
    ("Sok", "inne", 9.0),
    ("Czekolada na goraco", "inne", 12.0),
]

METODY_PLATNOSCI = ["karta", "gotowka", "blik"]
METODY_WAGI = [0.55, 0.20, 0.25]

N_TRANSAKCJI = 100_000
START_DATA = dt.datetime(2025, 11, 1)

sklepy_ids = sklepy_pdf["sklep"].tolist()

rows = []
for tid in range(1, N_TRANSAKCJI + 1):
    sklep = random.choice(sklepy_ids)
    miasto = sklep_do_miasta[sklep]
    produkt, kategoria, cena_bazowa = random.choice(PRODUKTY)

    # więcej ruchu rano (7-10) i popoludniu (12-15) niż wieczorem
    godzina = int(np.clip(np.random.normal(loc=12, scale=3.5), 6, 21))
    dzien_offset = random.randint(0, 364)
    minuta = random.randint(0, 59)
    data_czas = START_DATA + dt.timedelta(days=dzien_offset, hours=godzina, minutes=minuta)

    cena = round(cena_bazowa + np.random.normal(0, 0.5), 2)
    ilosc = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
    metoda_platnosci = random.choices(METODY_PLATNOSCI, weights=METODY_WAGI)[0]

    rows.append((
        tid, data_czas, sklep, miasto, produkt, kategoria,
        max(cena, 1.0), ilosc, metoda_platnosci,
    ))

kolumny = [
    "transakcja_id", "data_czas", "sklep", "miasto", "produkt",
    "kategoria", "cena", "ilosc", "metoda_platnosci",
]
transakcje_pdf = pd.DataFrame(rows, columns=kolumny)

transakcje = spark.createDataFrame(transakcje_pdf)
transakcje = transakcje.withColumn("data_czas", F.col("data_czas").cast("timestamp"))

print(f"📊 Liczba transakcji: {transakcje.count()}")
transakcje.printSchema()
transakcje.show(5)

## 🎓 Guided Example: przychód per miasto

**Zadanie:** Oblicz łączny przychód (`cena * ilosc`) w podziale na miasto, posortowany malejąco.

Zobaczysz to samo zadanie napisane **dwoma sposobami** — Spark SQL i DataFrame API — żeby zobaczyć, że dają identyczny wynik.

In [ ]:
# @title 🎓 Guided Example — Run and study this

# Krok 1: rejestrujemy tymczasowy widok
transakcje.createOrReplaceTempView("transakcje")
sklepy.createOrReplaceTempView("sklepy")

# Krok 2: to samo zadanie — Spark SQL
przychod_sql = spark.sql("""
    SELECT miasto, ROUND(SUM(cena * ilosc), 2) AS przychod
    FROM transakcje
    GROUP BY miasto
    ORDER BY przychod DESC
""")

print("💬 Wynik SQL:")
przychod_sql.show()

# Krok 3: to samo zadanie — DataFrame API
przychod_api = (
    transakcje
    .withColumn("wartosc", F.col("cena") * F.col("ilosc"))
    .groupBy("miasto")
    .agg(F.round(F.sum("wartosc"), 2).alias("przychod"))
    .orderBy(F.col("przychod").desc())
)

print("🐍 Wynik DataFrame API:")
przychod_api.show()

---
## ✏️ Ćwiczenie 1: Najlepiej sprzedający się produkt per sklep

**Trudność:** ⭐⭐

Dla każdego sklepu znajdź produkt, który wygenerował **najwyższy przychód** (`cena * ilosc` zsumowane).

Wynikowy DataFrame `najlepszy_produkt` powinien mieć kolumny: `sklep`, `produkt`, `przychod` — dokładnie **jeden wiersz na sklep**.

**Oczekiwany rezultat:** DataFrame z 15 wierszami (po jednym na sklep).

> 💡 **Hint:** Policz przychód per `sklep` + `produkt`, a potem użyj `Window.partitionBy("sklep").orderBy(...)` z `F.row_number()`, żeby wybrać wiersz numer 1 w każdej grupie.

In [ ]:
# ✏️ Ćwiczenie 1: Najlepiej sprzedający się produkt per sklep

# TODO: policz przychód per sklep + produkt
przychod_produkt_sklep = ...  # Replace ... with your code

# TODO: zdefiniuj okno partycjonowane po sklepie, posortowane malejąco po przychodzie
okno_sklep = ...  # Replace ... with your code

# TODO: dodaj kolumnę z rankingiem i zostaw tylko ranga == 1
najlepszy_produkt = ...  # Replace ... with your code

najlepszy_produkt.show()

In [ ]:
# @title ✅ Test — Ćwiczenie 1 (Run to check your answer)

try:
    assert najlepszy_produkt is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    liczba_wierszy = najlepszy_produkt.count()
    assert liczba_wierszy == 15, f"❌ Oczekiwano 15 wierszy (po jednym na sklep), otrzymano {liczba_wierszy}"
    kolumny_wyniku = set(najlepszy_produkt.columns)
    assert {"sklep", "produkt", "przychod"}.issubset(kolumny_wyniku), "❌ Brakuje wymaganych kolumn"
    liczba_unikalnych_sklepow = najlepszy_produkt.select("sklep").distinct().count()
    assert liczba_unikalnych_sklepow == 15, "❌ Każdy sklep powinien wystąpić dokładnie raz"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 2: Przychód per kategoria i godzina dnia

**Trudność:** ⭐⭐

Chcemy zobaczyć, o której porze dnia sprzedaje się najwięcej poszczególnych kategorii produktów.

1. Dodaj kolumnę `godzina` wyciągniętą z `data_czas` (użyj `F.hour()`)
2. Stwórz tabelę przestawną (`pivot`) — wiersze to `godzina`, kolumny to `kategoria`, wartości to suma przychodu

**Oczekiwany rezultat:** DataFrame z kolumną `godzina` oraz osobną kolumną dla każdej kategorii (`kawa`, `herbata`, `przekaska`, `inne`).

> 💡 **Hint:** `.groupBy("godzina").pivot("kategoria").agg(F.sum(...))` — pamiętaj o dodaniu kolumny `wartosc` przed pivotem!

In [ ]:
# ✏️ Ćwiczenie 2: Przychód per kategoria i godzina dnia

# TODO: dodaj kolumnę 'godzina' oraz 'wartosc' (cena * ilosc)
transakcje_z_godzina = ...  # Replace ... with your code

# TODO: pivot — wiersze: godzina, kolumny: kategoria, wartości: suma(wartosc)
przychod_godzina_kategoria = ...  # Replace ... with your code

przychod_godzina_kategoria.orderBy("godzina").show()

In [ ]:
# @title ✅ Test — Ćwiczenie 2 (Run to check your answer)

try:
    assert przychod_godzina_kategoria is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert "godzina" in przychod_godzina_kategoria.columns, "❌ Brakuje kolumny 'godzina'"
    kategorie_oczekiwane = {"kawa", "herbata", "przekaska", "inne"}
    assert kategorie_oczekiwane.issubset(set(przychod_godzina_kategoria.columns)), \
        "❌ Brakuje kolumn dla wszystkich kategorii po pivocie"
    liczba_godzin = przychod_godzina_kategoria.select("godzina").distinct().count()
    assert liczba_godzin > 1, "❌ Powinno być więcej niż jedna wartość godziny"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 3: Wzrost przychodu miesiąc do miesiąca

**Trudność:** ⭐⭐⭐

Dla całej sieci kawiarni policz przychód **per miesiąc**, a następnie dodaj kolumnę pokazującą **procentowy wzrost** względem poprzedniego miesiąca.

**Oczekiwany rezultat:** DataFrame `wzrost_mom` z kolumnami `miesiac`, `przychod`, `przychod_poprzedni_miesiac`, `wzrost_procent`, posortowany rosnąco po `miesiac`.

> 💡 **Hint:** Wyciągnij miesiąc przez `F.date_format("data_czas", "yyyy-MM")`, zsumuj przychód, a potem użyj `F.lag()` na oknie posortowanym po miesiącu (bez `partitionBy`, bo interesuje nas cała sieć).

In [ ]:
# ✏️ Ćwiczenie 3: Wzrost przychodu miesiąc do miesiąca

# TODO: dodaj kolumnę 'miesiac' (format 'yyyy-MM') oraz 'wartosc' (cena * ilosc)
transakcje_z_miesiacem = ...  # Replace ... with your code

# TODO: zagreguj przychód per miesiąc
przychod_miesieczny = ...  # Replace ... with your code

# TODO: zdefiniuj okno posortowane po miesiącu (bez partitionBy)
okno_czas = ...  # Replace ... with your code

# TODO: dodaj kolumnę z przychodem z poprzedniego miesiąca (F.lag) oraz wzrost procentowy
wzrost_mom = ...  # Replace ... with your code

wzrost_mom.orderBy("miesiac").show()

In [ ]:
# @title ✅ Test — Ćwiczenie 3 (Run to check your answer)

try:
    assert wzrost_mom is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    wymagane_kolumny = {"miesiac", "przychod", "przychod_poprzedni_miesiac", "wzrost_procent"}
    assert wymagane_kolumny.issubset(set(wzrost_mom.columns)), \
        f"❌ Brakuje kolumn: {wymagane_kolumny - set(wzrost_mom.columns)}"
    liczba_miesiecy = wzrost_mom.count()
    assert liczba_miesiecy >= 10, f"❌ Oczekiwano danych z ~12 miesięcy, otrzymano {liczba_miesiecy}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 4: Join z tabelą `sklepy` — przychód per region

**Trudność:** ⭐⭐

Tabela `sklepy` zawiera informacje o regionie każdego sklepu. Połącz `transakcje` z `sklepy` i oblicz łączny przychód **per region**.

**Oczekiwany rezultat:** DataFrame `przychod_region` z kolumnami `region`, `przychod`, posortowany malejąco po przychodzie.

> 💡 **Hint:** Tabela `sklepy` ma tylko 15 wierszy — to świetny kandydat na `F.broadcast()`!

In [ ]:
# ✏️ Ćwiczenie 4: Join z tabelą sklepy — przychód per region

# TODO: połącz transakcje ze sklepami po kolumnie 'sklep' (użyj F.broadcast na sklepy)
transakcje_ze_sklepami = ...  # Replace ... with your code

# TODO: zagreguj przychód (cena * ilosc) per region i posortuj malejąco
przychod_region = ...  # Replace ... with your code

przychod_region.show()

In [ ]:
# @title ✅ Test — Ćwiczenie 4 (Run to check your answer)

try:
    assert przychod_region is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert "region" in przychod_region.columns, "❌ Brakuje kolumny 'region'"
    liczba_regionow = przychod_region.count()
    assert liczba_regionow == 5, f"❌ Oczekiwano 5 regionów, otrzymano {liczba_regionow}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 5: Najpopularniejsza metoda płatności per miasto

**Trudność:** ⭐⭐⭐

Dla każdego miasta znajdź **najczęściej używaną** metodę płatności (najwyższa liczba transakcji).

**Oczekiwany rezultat:** DataFrame `top_metoda_platnosci` z kolumnami `miasto`, `metoda_platnosci`, `liczba_transakcji` — dokładnie jeden wiersz na miasto.

> 💡 **Hint:** Podobnie jak w Ćwiczeniu 1 — zlicz transakcje per `miasto` + `metoda_platnosci`, a potem użyj okna z `F.row_number()`.

In [ ]:
# ✏️ Ćwiczenie 5: Najpopularniejsza metoda płatności per miasto

# TODO: policz liczbę transakcji per miasto + metoda_platnosci
liczba_transakcji_miasto_metoda = ...  # Replace ... with your code

# TODO: zdefiniuj okno partycjonowane po mieście, posortowane malejąco po liczbie transakcji
okno_miasto = ...  # Replace ... with your code

# TODO: dodaj ranking i zostaw tylko ranga == 1
top_metoda_platnosci = ...  # Replace ... with your code

top_metoda_platnosci.show()

In [ ]:
# @title ✅ Test — Ćwiczenie 5 (Run to check your answer)

try:
    assert top_metoda_platnosci is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    liczba_miast = top_metoda_platnosci.select("miasto").distinct().count()
    assert liczba_miast == 5, f"❌ Oczekiwano 5 miast, otrzymano {liczba_miast}"
    liczba_wierszy = top_metoda_platnosci.count()
    assert liczba_wierszy == 5, "❌ Powinien być dokładnie jeden wiersz na miasto"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 6: Zapytanie SQL z `HAVING`

**Trudność:** ⭐⭐

Napisz zapytanie `spark.sql()`, które zwraca sklepy (`sklep`), których **łączny przychód przekracza 40 000 PLN**.

**Oczekiwany rezultat:** DataFrame `sklepy_powyzej_progu` z kolumnami `sklep`, `przychod`, posortowany malejąco po przychodzie.

> 💡 **Hint:** `GROUP BY sklep HAVING SUM(cena * ilosc) > 40000` — pamiętaj, że `HAVING` filtruje **po** agregacji, w przeciwieństwie do `WHERE`.

In [ ]:
# ✏️ Ćwiczenie 6: Zapytanie SQL z HAVING

# TODO: napisz zapytanie spark.sql() z GROUP BY ... HAVING
sklepy_powyzej_progu = ...  # Replace ... with your code

sklepy_powyzej_progu.show()

In [ ]:
# @title ✅ Test — Ćwiczenie 6 (Run to check your answer)

try:
    assert sklepy_powyzej_progu is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert {"sklep", "przychod"}.issubset(set(sklepy_powyzej_progu.columns)), \
        "❌ Brakuje wymaganych kolumn ('sklep', 'przychod')"
    progi = [wiersz["przychod"] for wiersz in sklepy_powyzej_progu.collect()]
    assert all(p > 40000 for p in progi), "❌ Wszystkie przychody powinny przekraczać 40000"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 7: Filtrowanie zaawansowane — poranny ruch

**Trudność:** ⭐⭐

Zarząd chce wiedzieć, jaki **procent** wszystkich transakcji odbywa się **rano** (godziny 6:00–10:00, kategoria `kawa`, płatność `blik`).

1. Dodaj kolumnę `godzina`
2. Przefiltruj transakcje spełniające **wszystkie trzy** warunki jednocześnie
3. Oblicz procent takich transakcji względem wszystkich transakcji

**Oczekiwany rezultat:** zmienna `procent_porannych_blik` typu `float`.

> 💡 **Hint:** Połącz warunki operatorem `&` w `.filter()`, np. `(F.col("godzina") >= 6) & (F.col("godzina") < 10) & ...`. Pamiętaj o nawiasach wokół każdego warunku!

In [ ]:
# ✏️ Ćwiczenie 7: Filtrowanie zaawansowane — poranny ruch

# TODO: dodaj kolumnę 'godzina'
transakcje_godzina7 = ...  # Replace ... with your code

# TODO: przefiltruj: godzina 6-10, kategoria == 'kawa', metoda_platnosci == 'blik'
poranne_blik = ...  # Replace ... with your code

# TODO: oblicz procent (float) takich transakcji względem wszystkich transakcji
procent_porannych_blik = ...  # Replace ... with your code

print(f"Procent porannych transakcji kawa+blik: {procent_porannych_blik:.2f}%")

In [ ]:
# @title ✅ Test — Ćwiczenie 7 (Run to check your answer)

try:
    assert procent_porannych_blik is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert isinstance(procent_porannych_blik, float), "❌ Wynik powinien być typu float"
    assert 0.0 <= procent_porannych_blik <= 100.0, "❌ Procent powinien być między 0 a 100"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## 🏆 Bonus Challenge (Optional): Optymalizacja broadcast join

Dla chętnych! 🌟

1. Wykonaj join `transakcje` z `sklepy` **bez** `F.broadcast()` i wywołaj `.explain()` na wyniku
2. Wykonaj ten sam join **z** `F.broadcast(sklepy)` i również wywołaj `.explain()`
3. Porównaj oba plany wykonania — znajdź w planie z broadcastem krok `BroadcastExchange` / `BroadcastHashJoin`
4. Napisz w komórce Markdown (poniżej), jaką różnicę zauważyłeś/aś w planach wykonania

> 💡 Nawet jeśli Spark **sam** zdecydował się na broadcast (bo `sklepy` jest małą tabelą), warto zobaczyć to w planie na własne oczy!

In [ ]:
# 🏆 Bonus: porównanie planów wykonania z i bez broadcast

print("=== BEZ jawnego broadcast ===")
transakcje.join(sklepy, "sklep").explain()

print("\n=== Z jawnym F.broadcast() ===")
transakcje.join(F.broadcast(sklepy), "sklep").explain()

*(Twoje obserwacje na temat różnicy w planach wykonania — wpisz tutaj)*

---
## 📝 Podsumowanie

Świetna robota! 🎉 Oto co przećwiczyłeś/aś w tym laboratorium:

- ✅ Rejestrowanie tymczasowych widoków i pisanie zapytań `spark.sql()`
- ✅ Agregacje (`groupBy`, `agg`, `pivot`) i ich odpowiedniki SQL
- ✅ Window functions (`row_number`, `lag`) do rankingów i analizy trendów czasowych
- ✅ Joiny z tabelą wymiarową i optymalizacja przez `broadcast()`
- ✅ Filtrowanie z wieloma warunkami jednocześnie

### 📚 Dalsze materiały
- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [PySpark Window Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)

### 🔮 Co dalej?
Na **Wykładzie 6** poznamy **zaawansowane window functions** i **optymalizację zapytań** (Catalyst, `explain()`, caching) — czas zamienić się w prawdziwych baristów danych! ☕📊